In [ ]:
import h5py
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Путь к данным
project_root = Path.cwd().parent
data_path = project_root / "data" / "raw" / "Galaxy10_DECals.h5"

with h5py.File(data_path, 'r') as F:
    # ==========================================
    # 1. Сетка 4x4 со случайными изображениями
    # ==========================================
    n_images = F['images'].shape[0]
    indices = np.sort(np.random.choice(n_images, 16, replace=False))
    
    images_subset = F['images'][indices]
    labels_subset = F['ans'][indices]
    
    fig1, axes = plt.subplots(4, 4, figsize=(16, 16))
    
    for i, ax in enumerate(axes.flat):
        img = images_subset[i]
        label = int(labels_subset[i])
        idx = indices[i]
        
        ax.imshow(img)
        ax.set_title(f'#{idx}\nType: {label}', fontsize=11, fontweight='bold')
        ax.axis('off')

    fig1.tight_layout()
    
    # ==========================================
    # 2. Гистограмма распределения классов
    # ==========================================
    labels_all = F['ans'][:]
    unique, counts = np.unique(labels_all, return_counts=True)
    colors = plt.cm.tab10(np.arange(10))
    
    fig2, ax = plt.subplots(figsize=(12, 6))
    bars = ax.bar(unique, counts, color=colors[unique], edgecolor='black', linewidth=1)
    
    ax.set_xlabel('Тип галактики', fontsize=13)
    ax.set_ylabel('Кол-во картинок', fontsize=13)
    ax.set_title(f'Распределение типов галактик (Всего: {len(labels_all):,} изображений)', fontsize=15, fontweight='bold')
    ax.set_xticks(unique)
    ax.set_xticklabels([f'Тип {int(u)}' for u in unique], fontsize=11)
    ax.grid(axis='y', alpha=0.7, linestyle='--')
    ax.set_axisbelow(True)
    
    for bar, count in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 35,
                f'{count}', ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    fig2.tight_layout()

plt.show()

In [7]:
import json
import h5py
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split

# Путь к данным
project_root = Path.cwd().parent
data_path = project_root / "data" / "raw" / "Galaxy10_DECals.h5"

with h5py.File(data_path, 'r') as F:
    labels = F['ans'][:]
    indices = np.arange(len(labels))

    # Сначала train/temp
    train_idx, temp_idx = train_test_split(
        indices, 
        test_size=0.3, 
        stratify=labels,
        random_state=42
    )

    # Потом temp на val/test
    val_idx, test_idx = train_test_split(
        temp_idx,
        test_size=0.5,
        stratify=labels[temp_idx],
        random_state=42
    )

print(f"Train: {len(train_idx)}, Val: {len(val_idx)}, Test: {len(test_idx)}")

# ==========================================
# Сохранение индексов в JSON
# ==========================================

# Формируем структуру с метаданными
split_indices = {
    "meta": {
        "seed": 42,
        "train_size": 0.7,
        "val_size": 0.15,
        "test_size": 0.15,
        "created_from": "Galaxy10_DECals.h5"
    },
    "train": train_idx.tolist(),
    "val": val_idx.tolist(),
    "test": test_idx.tolist()
}

# Сохраняем в файл
save_path = project_root / "data" / "processed" / "indices_split.json"
save_path.parent.mkdir(parents=True, exist_ok=True)

with open(save_path, 'w') as f:
    json.dump(split_indices, f, indent=4)

Train: 12415, Val: 2660, Test: 2661
